# **Dynamic RAG**

In [28]:
import os
import huggingface_hub
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
!pip install datasets==2.20.0

In [3]:
!pip install transformers==4.41.2

In [5]:
!pip install accelerate==0.31.0

  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling 2.88.0 requires accelerate<2,>=1.0.0, but you have accelerate 0.31.0 which is incompatible.
docling-ibm-models 3.13.0 requires accelerate<2.0.0,>=1.2.1, but you have accelerate 0.31.0 which is incompatible.
docling-ibm-models 3.13.0 requires transformers!=5.0.*,!=5.1.*,!=5.2.*,!=5.3.*,<6.0.0,>=4.42.0, but you have transformers 4.41.2 which is incompatible.


In [4]:
from transformers import AutoTokenizer
import transformers
import torch

model = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model, use_auth_token=get_hf_token())
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
    use_auth_token=get_hf_token(),
)

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\auto\tokenization_auto.py:769: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
W0415 16:51:47.392000 104548 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\accelerate\utils\modeling.py:1384: UserWarning: Current model requires 318769536 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

### Chroma

In [15]:
!pip install chromadb==0.5.3

   ---------------------------------------- 0.0/559.5 kB ? eta -:--:--
   ---------------------------------------- 559.5/559.5 kB 18.6 MB/s  0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ----------------------------- ---------- 11.5/15.8 MB 51.5 MB/s eta 0:00:01
   ---------------------------------------- 15.8/15.8 MB 52.4 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 2.2.6

   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
    Uninstalling numpy-2.2.6:
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   -----------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling 2.88.0 requires accelerate<2,>=1.0.0, but you have accelerate 0.31.0 which is incompatible.
docling-ibm-models 3.13.0 requires accelerate<2.0.0,>=1.2.1, but you have accelerate 0.31.0 which is incompatible.
docling-ibm-models 3.13.0 requires transformers!=5.0.*,!=5.1.*,!=5.2.*,!=5.3.*,<6.0.0,>=4.42.0, but you have transformers 4.41.2 which is incompatible.
llama-index-vector-stores-deeplake 0.5.0 requires llama-index-core<0.15,>=0.13.0, but you have llama-index-core 0.10.68.post1 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [16]:
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------- ----------------------------- 8.4/33.5 MB 43.5 MB/s eta 0:00:01
     ------------------- ------------------- 16.5/33.5 MB 40.0 MB/s eta 0:00:01
     ---------------------------- ---------- 24.1/33.5 MB 39.2 MB/s eta 0:00:01
     --------------------------------------  32.8/33.5 MB 39.3 MB/s eta 0:00:01
     ---------------------------------------- 33.5/33.5 MB 36.1 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [5]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("sciq", split="train")
filtered_dataset = dataset.filter(lambda x: x["support"] != "" and x["correct_answer"] != "")
print(f"Number of samples after filtering: {len(filtered_dataset)}")

Number of samples after filtering: 10481


In [6]:
df = pd.DataFrame(filtered_dataset)
columns_to_drop = ['distractor3', 'distractor1', 'distractor2']
df.drop(columns=columns_to_drop, inplace=True)
df['completion'] = df['correct_answer'] + " because " + df['support']
df.dropna(subset=['completion'], inplace=True)
df

,question,correct_answer,support,completion
0,What type of organism is commonly used in prep...,mesophilic organisms,"Mesophiles grow best in moderate temperature, ...",mesophilic organisms because Mesophiles grow b...
1,What phenomenon makes global winds blow northe...,coriolis effect,Without Coriolis Effect the global winds would...,coriolis effect because Without Coriolis Effec...
2,Changes from a less-ordered state to a more-or...,exothermic,Summary Changes of state are examples of phase...,exothermic because Summary Changes of state ar...
3,What is the least dangerous radioactive decay?,alpha decay,All radioactive decay is dangerous to living t...,alpha decay because All radioactive decay is d...
4,Kilauea in hawaii is the world’s most continuo...,smoke and ash,Example 3.5 Calculating Projectile Motion: Hot...,smoke and ash because Example 3.5 Calculating ...
...,...,...,...,...
10476,The enzyme pepsin plays an important role in t...,peptides,Protein A large part of protein digestion take...,peptides because Protein A large part of prote...
10477,What remains a constant of radioactive substan...,rate of decay,The rate of decay of a radioactive substance i...,rate of decay because The rate of decay of a r...
10478,"Terrestrial ecosystems, also known for their d...",biomes,"Terrestrial ecosystems, also known for their d...","biomes because Terrestrial ecosystems, also kn..."
10479,High explosives create shock waves that exceed...,supersonic,The modern day formulation of gun powder is ca...,supersonic because The modern day formulation ...


In [7]:
df.shape

(10481, 4)

In [8]:
print(df.columns)

Index(['question', 'correct_answer', 'support', 'completion'], dtype='object')


### Embedding and upserting the data in a Chroma collection


In [9]:
!pip install numpy==1.26.4

In [10]:
import chromadb

client = chromadb.Client()

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [11]:
collection_name="sciq_supports6"
collections = client.list_collections()
collection_exists = any(collection.name == collection_name for collection in collections)
print(f"Collection '{collection_name}' exists: {collection_exists}")

Collection 'sciq_supports6' exists: False


In [12]:
if not collection_exists:
    collection = client.create_collection(name=collection_name)
else:
    print(f"Collection '{collection_name}' already exists.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [13]:
results = collection.get()
for result in results:
    print(result)

ids
embeddings
metadatas
documents
uris
data
included


In [14]:
model_name = "all-MiniLM-L6-v2"

In [15]:
ldf=len(df)

In [17]:
nb=ldf  # number of questions to embed and store
import time
start_time = time.time()  # Start timing before the request
batch_size = 5000
# Convert Series to list of strings
completion_list = df["completion"][:nb].astype(str).tolist()

# Avoiding trying to load data twice in this one run dynamic RAG notebook
if collection_exists!=True:
  # Embed and store the first nb supports for this demo
  for start in range(0, nb, batch_size):
    end = min(start + batch_size, nb)
    collection.add(
        ids=[str(i) for i in range(start, end)],
        documents=completion_list[start:end],
        metadatas=[{"type": "completion"} for _ in range(start, end)],
    )

response_time = time.time() - start_time  # Measure response time
print(f"Response Time: {response_time:.2f} seconds")  # Print response time

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Response Time: 503.99 seconds


In [18]:
result = collection.get(include=['embeddings'])
first_embedding = result['embeddings'][0]
embedding_length = len(first_embedding)
print(f"Length of the first embedding: {embedding_length}")
print(f"Total number of embeddings in the collection: {len(result['embeddings'])}")
print(f"Embedding of the first document: {first_embedding}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Length of the first embedding: 384
Total number of embeddings in the collection: 10481
Embedding of the first document: [0.03689070791006088, -0.05881566181778908, -0.048181332647800446, 0.06923316419124603, 0.016696497797966003, -0.040753722190856934, 0.018839966505765915, 0.018102362751960754, 0.01780514232814312, 0.07787054777145386, 0.025281663984060287, -0.15792308747768402, -0.023618163540959358, 0.09529945999383926, -0.005831793881952763, -0.009351729415357113, 0.08793967217206955, -0.02978258766233921, -0.031759634613990784, 0.0003584769438020885, 0.04816021770238876, 0.035945598036050797, -0.06368855386972427, -0.03580130264163017, 0.008479475975036621, -0.04704919457435608, -0.014411584474146366, 0.015326163731515408, -0.017449261620640755, 0.03771509230136871, -0.05390031263232231, 0.0012938095023855567, 0.1407582312822342, -0.012112556956708431, 0.016001125797629356, 0.025889595970511436, 0.009293322451412678, -0.1314585655927658, 0.04734911769628525, 0.05548204854130745, -

In [19]:
result = collection.get(include=["documents"])
first_document = result['documents'][0]
print(f"First document in the collection: {first_document}")

First document in the collection: mesophilic organisms because Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.


### Querying the collection

In [20]:
import time
start_time = time.time()  # Start timing before the request

results = collection.query(
    query_texts=df["question"][:nb],
    n_results=1
)

response_time = time.time() - start_time  # Measure response time
print(f"Response Time: {response_time:.2f} seconds")  # Print response

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Response Time: 424.28 seconds


In [21]:
import spacy
import numpy as np

# Load the pre-trained spaCy language model
nlp = spacy.load('en_core_web_md')  # Ensure that you've installed this model with 'python -m spacy download en_core_web_md'

def simple_text_similarity(text1, text2):
    doc1 = nlp(text1)
    doc2 = nlp(text2)

    # Get the vectors for each document
    vector1 = doc1.vector
    vector2 = doc2.vector
    if np.linalg.norm(vector1) == 0 or np.linalg.norm(vector2) == 0:
        return 0.0  
    else:
        similarity = np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))
        return similarity

In [22]:
nbqd = 100  # the number of responses to display supposing there are more than 100 records

# Print the question, the original completion, the retrieved document, and compare them
acc_counter=0
display_counter=0
for i, q in enumerate(df['question'][:nb]):
    original_completion = df['completion'][i]  
    retrieved_document = results['documents'][i][0]  # Retrieve the corresponding document
    similarity_score = simple_text_similarity(original_completion, retrieved_document)
    if similarity_score > 0.7:
      acc_counter+=1
    display_counter+=1
    if display_counter<=nbqd or display_counter>nb-nbqd:
      print(i," ", f"Question: {q}")
      print(f"Retrieved document: {retrieved_document}")
      print(f"Original completion: {original_completion}")
      print(f"Similarity Score: {similarity_score:.2f}")
      print()  

if nb>0:
  acc=acc_counter/nb
  print(f"Number of documents: {nb:.2f}")
  print(f"Overall similarity score: {acc:.2f}")

0   Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Retrieved document: lactic acid because Bacteria can be used to make cheese from milk. The bacteria turn the milk sugars into lactic acid. The acid is what causes the milk to curdle to form cheese. Bacteria are also involved in producing other foods. Yogurt is made by using bacteria to ferment milk ( Figure below ). Fermenting cabbage with bacteria produces sauerkraut.
Original completion: mesophilic organisms because Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.
Similarity Score: 0.88

1   Question: What phenomenon makes global winds blow nor

### Prompt and retrieval

In [23]:
prompt = "Millions of years ago, plants used energy from the sun to form what?"

In [24]:
import time
import textwrap

start_time = time.time()

results = collection.query(
    query_texts=[prompt],  
    n_results=1 
)
response_time = time.time() - start_time
print(f"Response Time: {response_time:.2f} seconds\n")

if results['documents'] and len(results['documents'][0]) > 0:
    wrapped_question = textwrap.fill(prompt, width=70)  # Wrap text at 70 characters
    wrapped_document = textwrap.fill(results['documents'][0][0], width=70)

    print(f"Question: {wrapped_question}")
    print("\n")
    print(f"Retrieved document: {wrapped_document}")
    print()
else:
    print("No documents retrieved.")


Response Time: 0.41 seconds

Question: Millions of years ago, plants used energy from the sun to form what?


Retrieved document: sun because The Sun supports most of Earth's ecosystems. Plants create
chemical energy from abiotic factors that include solar energy. The
food energy created by producers is passed through the food chain.



# RAG with Lllama

In [25]:
def LLaMA2(prompt):
    sequences = pipeline(
        prompt,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=100,  # Control the output length more granularly
        temperature=0.5,  # Slightly higher for more diversity
        repetition_penalty=2.0,  # Adjust based on experimentation
        truncation=True
    )
    return sequences

In [26]:
iprompt='Read the following input and write a summary for beginners.'
lprompt=iprompt + " " + results['documents'][0][0]

In [27]:
import time
start_time = time.time()  

response=LLaMA2(lprompt)
for seq in response:
    generated_part = seq['generated_text'].replace(iprompt, '')  # Remove the input part from the output

response_time = time.time() - start_time  
print(f"Response Time: {response_time:.2f} seconds")  

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ValueError: The following `model_kwargs` are not used by the model: ['use_auth_token'] (note: typos in the generate arguments will also show up in this list)

In [ ]:
wrapped_response = textwrap.fill(generated_part.strip(), width=70)
print(wrapped_response)

# Delete collection

In [ ]:
delete_collection=False
if delete_collection==True:
  client.delete_collection(collection_name)

In [ ]:
collections = client.list_collections()

collection_exists = any(collection.name == collection_name for collection in collections)
print("Collection exists:", collection_exists)